# 01. End-to-End Smoke Test (6 mẫu VCTK US/UK -> Whisper -> MT -> VieNeu-TTS & viXTTS -> Eval)
Notebook này chạy thử toàn bộ pipeline từ đầu đến cuối trên **6 mẫu** (1 speaker US + 1 speaker UK $\times$ 3 câu = 6 mẫu, kèm tự động cắt reference `3s / 5s / 8s` clean & noisy):
- **Bước 1:** Stream 6 mẫu từ `sanchit-gandhi/vctk` (CC BY 4.0) và tạo reference clips.
- **Bước 2:** Nhận dạng giọng nói tiếng Anh với `faster-whisper-large-v3` + chấm WER.
- **Bước 3:** Dịch sang tiếng Việt với `vinai/vinai-translate-en2vi-v2`.
- **Bước 4:** Tổng hợp giọng nói tiếng Việt zero-shot với **VieNeu-TTS v3 Turbo** (48 kHz) và **viXTTS** (24 kHz).
- **Bước 5:** Chấm điểm tự động (**SIM-ECAPA**, **SIM-WavLM**, **UTMOSv2**, **CER PhoWhisper**, **Prosody F0/Speaking Rate**) và nghe thử trực tiếp trong notebook.


In [ ]:
import os, json
from pathlib import Path
from IPython.display import Audio, display
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    pass
REPO_DIR = Path('/content/NLP_project') if Path('/content/NLP_project').exists() else Path.cwd()
if REPO_DIR.name == 'notebooks':
    REPO_DIR = REPO_DIR.parent
os.chdir(REPO_DIR)
!pip install -e . --no-deps -q


In [ ]:
# Bước 1: Chuẩn bị 6 mẫu VCTK (1 US + 1 UK x 3 câu) + cắt reference 3s/5s/8s (clean & noisy SNR=10dB)
!python scripts/run_stage.py --stage prepare_vctk --config configs/data/vctk.yaml --run-id smoke_e2e --smoke


In [ ]:
# Bước 2: Chạy STT (faster-whisper large-v3) + Tính WER theo accent
!python scripts/run_stage.py --stage stt --config configs/stt/faster_whisper_large_v3.yaml \
    --run-id smoke_e2e \
    --input-manifest processed/manifests/vctk_smoke.jsonl \
    --output-manifest outputs/stt/vctk_smoke_stt.jsonl

!python scripts/run_eval.py --mode wer \
    --input-manifest outputs/stt/vctk_smoke_stt.jsonl \
    --output-file metrics/smoke_e2e/wer_summary.json


In [ ]:
# Bước 3: Dịch máy EN -> VI bằng `vinai/vinai-translate-en2vi-v2`
!python scripts/run_stage.py --stage mt --config configs/mt/vinai_v2.yaml \
    --run-id smoke_e2e \
    --input-manifest outputs/stt/vctk_smoke_stt.jsonl \
    --output-manifest outputs/mt/vctk_smoke_vinai.jsonl


In [ ]:
# Bước 4A: Tổng hợp giọng tiếng Việt bằng VieNeu-TTS v3 Turbo (cloning 8s clean + preset anchor)
!python scripts/run_stage.py --stage tts --config configs/tts/vieneu_v3turbo.yaml \
    --run-id smoke_e2e \
    --input-manifest outputs/mt/vctk_smoke_vinai.jsonl \
    --output-manifest outputs/tts/vctk_smoke_vieneu_clone8s.jsonl \
    --condition-tag clone_8s_clean_denoise_on --ref-duration 8s --ref-quality clean

# Chạy thêm Preset Voice Anchor (giọng Việt bản ngữ) để làm mốc so sánh SIM & Accentedness
!python scripts/run_stage.py --stage tts --config configs/tts/vieneu_v3turbo.yaml \
    --run-id smoke_e2e \
    --input-manifest outputs/mt/vctk_smoke_vinai.jsonl \
    --output-manifest outputs/tts/vctk_smoke_vieneu_anchor.jsonl \
    --condition-tag preset_native_anchor --preset-anchor


In [ ]:
# Bước 4B (Tuỳ chọn): Tổng hợp với Baseline `capleaf/viXTTS`
!python scripts/run_stage.py --stage tts --config configs/tts/vixtts.yaml \
    --run-id smoke_e2e \
    --input-manifest outputs/mt/vctk_smoke_vinai.jsonl \
    --output-manifest outputs/tts/vctk_smoke_vixtts_clone8s.jsonl \
    --condition-tag clone_8s_clean --ref-duration 8s --ref-quality clean


In [ ]:
# Bước 5: Chấm điểm tự động (SIM-ECAPA, SIM-WavLM, UTMOSv2, CER PhoWhisper, Prosody)
!python scripts/run_eval.py --mode audio \
    --input-manifest outputs/tts/vctk_smoke_vieneu_clone8s.jsonl \
    --output-file metrics/smoke_e2e/eval_vieneu_clone8s.jsonl

!python scripts/run_eval.py --mode audio \
    --input-manifest outputs/tts/vctk_smoke_vieneu_anchor.jsonl \
    --output-file metrics/smoke_e2e/eval_vieneu_anchor.jsonl


In [ ]:
# Bước 6: Xem bảng tổng hợp & nghe thử mẫu đầu tiên ngay trong Notebook
from s2st.io import ProjectPaths
from s2st.manifest import read_manifest
paths = ProjectPaths('configs/paths.yaml')
eval_recs = read_manifest(paths.drive_root / 'metrics/smoke_e2e/eval_vieneu_clone8s.jsonl')
if eval_recs:
    r0 = eval_recs[0]
    print(f"Utterance: {r0['utt_id']} | Speaker: {r0['speaker_id']} ({r0['accent']})")
    print(f"EN (Gold)   : {r0['gold_en_text']}")
    print(f"EN (Whisper): {r0['stt_en_text']}")
    print(f"VI (MT)     : {r0['mt_vi_text']}")
    print(f"SIM-ECAPA (heldout): {r0.get('sim_ecapa_heldout')} | UTMOS: {r0.get('utmos')} | VI CER: {r0.get('vi_cer')}")
    print('\n--- Reference Audio (English 8s) ---')
    display(Audio(r0['tts_used_ref_audio_path']))
    print('--- Cloned Vietnamese Output (VieNeu-TTS v3 Turbo) ---')
    display(Audio(r0['tts_audio_path']))
